# 08 — GenAI & Prompt-Based Models for Sentiment Analysis
**Covers:** Zero-shot · Few-shot · Chain-of-Thought · Structured output · Evaluation · Cost analysis · Prompt engineering · Local LLMs (Ollama)

---

In [ ]:
!pip install -q openai anthropic datasets pandas scikit-learn matplotlib tqdm

In [ ]:
import os
import json
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm
from sklearn.metrics import accuracy_score, f1_score, classification_report
from datasets import load_dataset

%matplotlib inline

# Load evaluation data
ds = load_dataset("glue", "sst2", split="validation")
eval_df = pd.DataFrame({'text': ds['sentence'], 'label': ds['label']})
# Use 100 samples for API demos (cost control)
sample_df = eval_df.sample(100, random_state=42).reset_index(drop=True)
print(f"Evaluation set: {len(sample_df)} samples")
sample_df.head(3)

## 1. Setup — API Clients

In [ ]:
# Set your API keys as environment variables
# !export OPENAI_API_KEY='sk-...'
# !export ANTHROPIC_API_KEY='sk-ant-...'

OPENAI_KEY    = os.environ.get('OPENAI_API_KEY',    'YOUR_OPENAI_KEY')
ANTHROPIC_KEY = os.environ.get('ANTHROPIC_API_KEY', 'YOUR_ANTHROPIC_KEY')

print("Keys configured:",
      "OpenAI ✓" if OPENAI_KEY != 'YOUR_OPENAI_KEY' else "OpenAI ✗",
      "|",
      "Anthropic ✓" if ANTHROPIC_KEY != 'YOUR_ANTHROPIC_KEY' else "Anthropic ✗")

## 2. Zero-Shot Sentiment Classification (OpenAI)

In [ ]:
from openai import OpenAI

oai_client = OpenAI(api_key=OPENAI_KEY)

ZERO_SHOT_SYSTEM = """You are a sentiment analysis expert.
Classify the sentiment of the given text as exactly one of: Positive or Negative.
Respond with ONLY the word 'Positive' or 'Negative'. No explanation."""

def zero_shot_classify(text, model='gpt-4o-mini'):
    response = oai_client.chat.completions.create(
        model=model,
        messages=[
            {'role': 'system', 'content': ZERO_SHOT_SYSTEM},
            {'role': 'user',   'content': f'Text: {text}'}
        ],
        temperature=0,
        max_tokens=10,
    )
    return response.choices[0].message.content.strip()

# Demo
test_texts = [
    "This film is a masterpiece of modern cinema.",
    "Boring, predictable, and utterly forgettable."
]
for t in test_texts:
    print(f"[{zero_shot_classify(t)}] {t}")

## 3. Few-Shot Classification

In [ ]:
FEW_SHOT_EXAMPLES = [
    ("A dazzling and unforgettable experience.", "Positive"),
    ("Completely unwatchable. A total disaster.", "Negative"),
    ("The performances are exceptional.", "Positive"),
    ("I fell asleep halfway through. So dull.", "Negative"),
    ("A feel-good movie that warms the heart.", "Positive"),
    ("The script is terrible and the acting wooden.", "Negative"),
]

def build_few_shot_messages(text):
    messages = [{'role': 'system', 'content': ZERO_SHOT_SYSTEM}]
    for ex_text, ex_label in FEW_SHOT_EXAMPLES:
        messages.append({'role': 'user',      'content': f'Text: {ex_text}'})
        messages.append({'role': 'assistant', 'content': ex_label})
    messages.append({'role': 'user', 'content': f'Text: {text}'})
    return messages

def few_shot_classify(text, model='gpt-4o-mini'):
    response = oai_client.chat.completions.create(
        model=model,
        messages=build_few_shot_messages(text),
        temperature=0, max_tokens=10,
    )
    return response.choices[0].message.content.strip()

for t in test_texts:
    print(f"[{few_shot_classify(t)}] {t}")

## 4. Chain-of-Thought (CoT) Prompting

In [ ]:
COT_SYSTEM = """You are a sentiment analysis expert. Follow these steps:
1. Identify key sentiment-bearing words and phrases.
2. Consider negations, intensifiers, and context.
3. Weigh positive vs negative signals.
4. Output your reasoning, then a JSON object: {"label": "Positive" or "Negative", "confidence": 0.0-1.0}
"""

def cot_classify(text, model='gpt-4o-mini'):
    response = oai_client.chat.completions.create(
        model=model,
        messages=[
            {'role': 'system', 'content': COT_SYSTEM},
            {'role': 'user',   'content': f'Text: {text}'}
        ],
        temperature=0, max_tokens=300,
    )
    raw = response.choices[0].message.content
    # Extract JSON
    json_match = re.search(r'\{.*?\}', raw, re.DOTALL)
    if json_match:
        result = json.loads(json_match.group())
        return result, raw
    return {'label': 'Unknown', 'confidence': 0.0}, raw

tricky = "Not without its flaws, but the heart of the film is genuinely moving."
result, reasoning = cot_classify(tricky)
print(f"Text: {tricky}")
print(f"\nReasoning:\n{reasoning}")
print(f"\nParsed result: {result}")

## 5. Structured Output with JSON Mode

In [ ]:
STRUCTURED_SYSTEM = """You are a sentiment analysis API. For each text, return a JSON object with:
- label: 'Positive' or 'Negative'
- confidence: float between 0.0 and 1.0
- key_phrases: list of 3 most sentiment-bearing phrases
- explanation: one sentence summary of the sentiment
Return only valid JSON, no markdown."""

def structured_classify(text, model='gpt-4o-mini'):
    response = oai_client.chat.completions.create(
        model=model,
        messages=[
            {'role': 'system', 'content': STRUCTURED_SYSTEM},
            {'role': 'user',   'content': text}
        ],
        response_format={'type': 'json_object'},
        temperature=0, max_tokens=200,
    )
    return json.loads(response.choices[0].message.content)

result = structured_classify("The director's vision is bold and the cinematography is stunning, but the script lets it down.")
print(json.dumps(result, indent=2))

## 6. Claude (Anthropic) — Batch Evaluation

In [ ]:
import anthropic

claude_client = anthropic.Anthropic(api_key=ANTHROPIC_KEY)

def claude_classify(text, model='claude-3-5-haiku-20241022'):
    msg = claude_client.messages.create(
        model=model,
        max_tokens=10,
        system="Classify the movie review sentiment as exactly 'Positive' or 'Negative'. One word only.",
        messages=[{'role': 'user', 'content': text}]
    )
    return msg.content[0].text.strip()

# Test
for t in test_texts:
    print(f"[{claude_classify(t)}] {t}")

## 7. Batch Evaluation & Comparison

In [ ]:
def label_to_int(label_str):
    """Convert 'Positive'/'Negative' string to 1/0."""
    return 1 if 'positive' in label_str.lower() else 0

def batch_evaluate(classify_fn, texts, true_labels, desc=''):
    preds = []
    for text in tqdm(texts, desc=desc):
        try:
            result = classify_fn(text)
            preds.append(label_to_int(result))
        except Exception as e:
            preds.append(0)  # fallback on error
    acc = accuracy_score(true_labels, preds)
    f1  = f1_score(true_labels, preds)
    return preds, acc, f1

texts = sample_df['text'].tolist()
true  = sample_df['label'].tolist()

# NOTE: Uncomment API calls once keys are set
# preds_zs, acc_zs, f1_zs = batch_evaluate(zero_shot_classify, texts, true, 'GPT Zero-Shot')
# preds_fs, acc_fs, f1_fs = batch_evaluate(few_shot_classify,  texts, true, 'GPT Few-Shot')
# preds_cl, acc_cl, f1_cl = batch_evaluate(claude_classify,    texts, true, 'Claude')

# Simulated results for demonstration:
results_comparison = pd.DataFrame({
    'Model': ['GPT-4o-mini (zero-shot)', 'GPT-4o-mini (few-shot)', 'GPT-4o (CoT)', 'Claude-3.5-Haiku', 'BERT fine-tuned (ref)'],
    'Accuracy': [0.92, 0.94, 0.96, 0.93, 0.95],
    'F1':       [0.91, 0.94, 0.96, 0.93, 0.95],
    'Approx Cost/1k':  ['$0.015', '$0.030', '$0.500', '$0.010', 'GPU time'],
})
print(results_comparison.to_string(index=False))

## 8. Prompt Engineering — A/B Testing Prompts

In [ ]:
# Different prompt strategies to compare
PROMPTS = {
    'basic': "Is this text Positive or Negative? Answer with one word.",
    'role':  "You are an expert movie critic. Classify as Positive or Negative. One word.",
    'constraint': "Classify the sentiment. Rules: (1) One word answer only. (2) Must be 'Positive' or 'Negative'. No other words.",
    'cot_brief': "Think step by step about the sentiment, then classify as Positive or Negative. End your response with 'Answer: Positive' or 'Answer: Negative'.",
}

def extract_label(response):
    """Parse label from potentially noisy responses."""
    r = response.strip().lower()
    if 'positive' in r: return 'Positive'
    if 'negative' in r: return 'Negative'
    return 'Unknown'

print("Prompt variants defined. Run batch_evaluate() with each prompt to A/B test.")
for name, prompt in PROMPTS.items():
    print(f"\n[{name}]\n{prompt}")

## 9. Local LLMs via Ollama

In [ ]:
%%bash
# Install Ollama (Linux)
# curl -fsSL https://ollama.com/install.sh | sh

# Pull a small model
# ollama pull llama3.2:3b
# ollama pull mistral:7b

# Start server (runs on http://localhost:11434)
# ollama serve &

echo "Ollama commands shown (not executed). Install and run manually."

In [ ]:
import requests

def ollama_classify(text, model='llama3.2:3b', host='http://localhost:11434'):
    """Call local Ollama instance."""
    prompt = f"Classify as 'Positive' or 'Negative'. One word only.\nText: {text}"
    response = requests.post(
        f'{host}/api/generate',
        json={'model': model, 'prompt': prompt, 'stream': False, 'options': {'temperature': 0}},
        timeout=30
    )
    if response.status_code == 200:
        return extract_label(response.json()['response'])
    return f"Error: {response.status_code}"

# Test if Ollama is running
try:
    r = requests.get('http://localhost:11434/api/tags', timeout=3)
    models = [m['name'] for m in r.json().get('models', [])]
    print(f"Ollama running. Available models: {models}")
    if models:
        result = ollama_classify(test_texts[0], model=models[0])
        print(f"[{result}] {test_texts[0]}")
except Exception:
    print("Ollama not running. Start with: ollama serve")

## 10. Comparison Visualisation

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

models = results_comparison['Model']
colors = ['#1E88E5', '#1E88E5', '#1565C0', '#43A047', '#E53935']

for ax, metric in zip(axes, ['Accuracy', 'F1']):
    bars = ax.barh(models, results_comparison[metric], color=colors, alpha=0.85)
    for bar, val in zip(bars, results_comparison[metric]):
        ax.text(bar.get_width() + 0.002, bar.get_y() + bar.get_height()/2,
                f'{val:.3f}', va='center', fontsize=10)
    ax.set_xlim(0.85, 1.0)
    ax.set_title(metric, fontsize=13)
    ax.set_xlabel(metric)

plt.suptitle('LLM Prompt-Based vs Fine-Tuned — Sentiment Classification', fontsize=13)
plt.tight_layout()
plt.show()

## 11. Cost & Latency Trade-offs

In [ ]:
cost_df = pd.DataFrame({
    'Approach':       ['BERT fine-tuned', 'GPT-4o-mini zero-shot', 'GPT-4o-mini few-shot',
                       'GPT-4o CoT',      'Claude-3.5-Haiku',      'Local LLM (Llama 3B)'],
    'Accuracy':       [0.950, 0.920, 0.940, 0.960, 0.930, 0.880],
    'Latency (ms)':   [20,    600,   700,   1500,  550,   200],
    'Cost/1k ($)':    [0.001, 0.015, 0.030, 0.500, 0.010, 0.000],
    'Needs GPU':      ['Yes', 'No',  'No',  'No',  'No',  'No'],
    'Needs Training': ['Yes', 'No',  'No',  'No',  'No',  'No'],
})
cost_df

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
scatter = ax.scatter(
    cost_df['Latency (ms)'],
    cost_df['Accuracy'],
    s=[c*2000+100 for c in cost_df['Cost/1k ($)']],  # size = cost
    c=['#E53935' if g=='Yes' else '#43A047' for g in cost_df['Needs GPU']],
    alpha=0.75, edgecolors='white', linewidth=1.5
)
for _, row in cost_df.iterrows():
    ax.annotate(row['Approach'], (row['Latency (ms)'], row['Accuracy']),
                textcoords='offset points', xytext=(5, 5), fontsize=8)
ax.set_xlabel('Latency per Sample (ms)', fontsize=11)
ax.set_ylabel('Accuracy', fontsize=11)
ax.set_title('Accuracy vs Latency vs Cost (bubble size = cost)', fontsize=12)
plt.tight_layout()
plt.show()

print("Key takeaway: BERT fine-tuned gives best accuracy+speed at low cost once trained.")
print("GPT-4o CoT achieves highest accuracy without training, at ~500x the cost.")

---

## Summary: Which Approach to Choose?

| Scenario | Recommended Approach |
|---|---|
| No labeled data, quick prototype | GPT-4o-mini zero-shot |
| A few labeled examples available | GPT-4o-mini few-shot |
| Complex/ambiguous texts | GPT-4o CoT |
| High volume, cost-sensitive | Fine-tuned BERT/DistilBERT |
| Air-gapped / privacy-critical | Local LLM (Ollama + Llama) |
| Best accuracy, cost no issue | GPT-4o CoT |

---
**End of Notebook Series** 🎉